# Part 0: Prerequisites
## Import neccessary libraries

In [1]:
import sys
sys.path.append('../src')

import numpy as np
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# plot and ML model
from sixth_meeting.visualizer import TransitionPointsVisualizer
from fifth_meeting.create_dataset import HammingDecayPredictor

print("All imports successful!")

All imports successful!


## Implement exponential decay function and a way to fit it to our data

In [2]:
# Fit hyperbolic decay curves to the smoothed data ((A / B * x) + C, A - initial value, B - decay rate, C - asymptotic value (bias))
def hyperbolic_decay(x, A, B, C):
    return (A / (B * x)) + C

def exponential_decay(x, A, B, C):
    return A * np.exp(-B * x) + C

def fit_curve_to_data(x_data, y_data, poly_type=exponential_decay):

    # Initial A, B, C guesses
    C_guess = np.mean(y_data[-5:])  # Estimate offset from last few points
    A_guess = np.max(y_data) - C_guess  # Amplitude from peak minus offset

    non_zero_mask = y_data - C_guess > 0.01
    if np.sum(non_zero_mask) > 2:
        log_y = np.log(y_data[non_zero_mask] - C_guess)
        x_for_fit = x_data[non_zero_mask]
        B_guess = -np.polyfit(x_for_fit, log_y, 1)[0]
    else:
        B_guess = 0.1  # Default guess

    initial_guess = [A_guess, B_guess, C_guess]

    # Apply Nonlinear Least Squares fitting for A, B, C
    try:
        # Basic fit
        params_opt, params_cov = curve_fit(
            poly_type, 
            x_data, 
            y_data,
            p0=initial_guess,
            maxfev=10000  # Increase max function evaluations
        )
        
        A_fit, B_fit, C_fit = params_opt
        perr = np.sqrt(np.diag(params_cov))  # Parameter uncertainties
        
    except RuntimeError as e:
        print(f"Optimization failed: {e}")
        print("Trying with bounds...")
        
        # Try with bounds to help convergence
        params_opt, params_cov = curve_fit(
            poly_type,
            x_data,
            y_data,
            p0=initial_guess,
            bounds=([0, 0, -np.inf], [np.inf, np.inf, np.inf]),  # A,B ≥ 0
            maxfev=10000
        )
        
        A_fit, B_fit, C_fit = params_opt

    # return predicted values and parameters
    return A_fit, B_fit, C_fit

## Load raw simulation data of the test circuits to display along with our Hellinger prediction

In [3]:
loaded = np.load(f'../data/sixth_meeting/test_circuits_simulation_dataset.npz', allow_pickle=True)
names_dataset = loaded['name']
circuits_info_dataset = loaded['circuit_info']
shots_dataset = loaded['shots_range']
noise_dataset = loaded['noise_range']
hamming_dataset = loaded['hamming_outputs']
hellinger_dataset = loaded['hellinger_outputs']

#--- Smooth the Hellinger and Hamming data using Savitzky-Golay filter
smoothed_hellinger = np.array([
            [savgol_filter(savgol_filter(hellinger_dataset[i][j], 11, 2), 11, 2)
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

smoothed_hamming = np.array([
            [[savgol_filter(savgol_filter(hamming_dataset[i][j][k], 11, 2), 11, 2)
              for k in range(len(noise_dataset[i]))]
             for j in range(10)]
            for i in range(len(names_dataset))
        ])

#--- Calculate the standard deviation of the smoothed Hamming data across the 10 runs
smoothed_hamming_std = [np.std(hamming_data, axis=0) for hamming_data in smoothed_hamming]


#--- Fit the smoothed Hellinger and Hamming data to the hyperbolic decay curve to extract A, B, C parameters
poly_hellinger_ABC = np.array([
                [fit_curve_to_data(shots_dataset[i], smoothed_hellinger[i][j])
                for j in range(len(noise_dataset[i]))]
                for i in range(len(names_dataset))
            ])

poly_hamming_std_ABC = np.array([
                [fit_curve_to_data(shots_dataset[i], smoothed_hamming_std[i][j])
                for j in range(len(noise_dataset[i]))]
                for i in range(len(names_dataset))
            ])

#--- Use the fitted A, B, C parameters to generate the fitted curves for Hellinger and Hamming data
poly_hellinger = np.array([
            [exponential_decay(shots_dataset[i], *poly_hellinger_ABC[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

poly_hamming_std = np.array([
            [exponential_decay(shots_dataset[i], *poly_hamming_std_ABC[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

# Part 1: Training our model
## Load training circuits data, normalize it, Create DataLoader

In [4]:
# load the training data
loaded_data = np.load('../data/sixth_meeting/ml_training_dataset.npz') # 90 circuits dataset
X_raw, Y_raw = loaded_data['X'], loaded_data['Y']

scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y_raw)

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_raw, dtype=torch.float32)     # [hamming_std on 50 and 70 shots, circuit supermarq features]
Y_tensor = torch.tensor(Y_scaled, dtype=torch.float32)  # [fitted A, B, C parameters for the Hellinger decay curve]


# Create DataLoader
dataset = TensorDataset(X_tensor, Y_tensor)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

## Train the model

In [5]:
### Initialize the HammingDecayPredictor model and train it
model = HammingDecayPredictor(input_dim=6, output_dim=3)

# MSE Loss is standard for regression. 
# Huber loss is less sensitive to outliers in data than MSE loss.
#criterion = nn.MSELoss()
criterion = nn.HuberLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

#--- Training loop
epochs = 1000
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch_X)
        
        # Compute loss
        loss = criterion(predictions, batch_Y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_X.size(0)
        
    total_epoch_loss = epoch_loss / len(train_loader.dataset)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_epoch_loss:.4f}")

Epoch [10/1000], Loss: 0.3923
Epoch [20/1000], Loss: 0.3674
Epoch [30/1000], Loss: 0.3490
Epoch [40/1000], Loss: 0.3403
Epoch [50/1000], Loss: 0.3318
Epoch [60/1000], Loss: 0.3270
Epoch [70/1000], Loss: 0.3210
Epoch [80/1000], Loss: 0.3158
Epoch [90/1000], Loss: 0.3124
Epoch [100/1000], Loss: 0.3059
Epoch [110/1000], Loss: 0.3023
Epoch [120/1000], Loss: 0.2961
Epoch [130/1000], Loss: 0.2918
Epoch [140/1000], Loss: 0.2883
Epoch [150/1000], Loss: 0.2842
Epoch [160/1000], Loss: 0.2814
Epoch [170/1000], Loss: 0.2740
Epoch [180/1000], Loss: 0.2713
Epoch [190/1000], Loss: 0.2657
Epoch [200/1000], Loss: 0.2612
Epoch [210/1000], Loss: 0.2580
Epoch [220/1000], Loss: 0.2550
Epoch [230/1000], Loss: 0.2463
Epoch [240/1000], Loss: 0.2414
Epoch [250/1000], Loss: 0.2370
Epoch [260/1000], Loss: 0.2375
Epoch [270/1000], Loss: 0.2301
Epoch [280/1000], Loss: 0.2262
Epoch [290/1000], Loss: 0.2242
Epoch [300/1000], Loss: 0.2197
Epoch [310/1000], Loss: 0.2202
Epoch [320/1000], Loss: 0.2146
Epoch [330/1000],

# Part 2: Perform Hellinger decay parameters prediction on unseen circuits
## Load the test circuits data and do a prediction for each of them

In [6]:
# load the test data
loaded_data = np.load('../data/sixth_meeting/ml_testing_dataset.npz')    # 18 circuits dataset
X_raw, _ = loaded_data['X'], loaded_data['Y']                                   # We don't need the Y values for testing, as we are predicting them

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_raw, dtype=torch.float32)

#--- Predict Hamming A, B, C parameters
model.eval()
with torch.no_grad():
    Y_pred_scaled = model(X_tensor)
    Y_pred = scaler_Y.inverse_transform(Y_pred_scaled.numpy())

# print(Y_pred.shape)

Y_reshaped = Y_pred.reshape(18, 1, 3)

## Visualize The Hamming STD, Actual Hellinger and its exponential fit, Predicted Hellinger and its exponential fit and display their saturation points

In [ ]:
slope_range = np.arange(0.00012, 0.0005, 0.00003) # a range of different slope thresholds

hellinger_points = []
predicted_hellinger_points = []

#--- Calculate the saturation points for every circuit, every noise and slope level in the specified range, and store them in a list to be plotted later
for circuit_idx in range(len(names_dataset)):
    hellinger_points_circuit = []
    predicted_hellinger_points_circuit = []
    for noise_idx in range(len(noise_dataset[circuit_idx])):
        hellinger_points_noise = []
        predicted_hellinger_points_noise = []
        for slope_threshold in slope_range:
            A, B, _ = poly_hellinger_ABC[circuit_idx][noise_idx]
            point_idx = np.where(np.abs(-A * B * np.exp(-B * shots_dataset[circuit_idx])) < slope_threshold)[0]
            hellinger_points_noise.append(point_idx[0])

            A_pred, B_pred, _ = Y_reshaped[circuit_idx * len(noise_dataset[circuit_idx]) + noise_idx][0]
            point_idx_pred = np.where(np.abs(-A_pred * B_pred * np.exp(-B_pred * shots_dataset[circuit_idx])) < slope_threshold)[0]
            predicted_hellinger_points_noise.append(point_idx_pred[0])
        hellinger_points_circuit.append(hellinger_points_noise)
        predicted_hellinger_points_circuit.append(predicted_hellinger_points_noise)
    hellinger_points.append(hellinger_points_circuit)
    predicted_hellinger_points.append(predicted_hellinger_points_circuit)


#--- Calculate the predicted Hellinger curves using the predicted A, B, C parameters from the ML model
poly_predicted_hellinger = np.array([
            [exponential_decay(shots_dataset[i], *Y_reshaped[i][j])
             for j in range(len(noise_dataset[i]))]
            for i in range(len(names_dataset))
        ])

#--- Visualize the results using the TransitionPointsVisualizer
exponential_decay_arrays = (poly_hellinger, poly_hamming_std, poly_predicted_hellinger)
saturation_points = (hellinger_points, predicted_hellinger_points)
visualizer_pred_hellinger = TransitionPointsVisualizer(names_dataset, shots_dataset, noise_dataset, smoothed_hellinger, smoothed_hamming_std, exponential_decay=exponential_decay_arrays, saturation_points=saturation_points)
visualizer_pred_hellinger.plot_graph_dashboard(slope_idx_init=6)

interactive(children=(Dropdown(description='Circuit:', options=(('varQC_linear_3r_r0', 0), ('varQC_linear_5r_r…

In [ ]:
def _error_metrics(actual, predicted, step):
    """Full metric dict for paired 1-D arrays of actual & predicted budgets."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    err = predicted - actual              # + overshoot, - undershoot
    abs_err = np.abs(err)
    n = len(actual)
 
    over, under, exact = err > 0, err < 0, err == 0
    ss_res = np.sum(err ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    if n > 1 and actual.std() > 0 and predicted.std() > 0:
        pearson = np.corrcoef(actual, predicted)[0, 1]
    else:
        pearson = float("nan")
 
    return {
        "n": n,
        "MAE_shots": abs_err.mean(),
        "median_abs_err_shots": float(np.median(abs_err)),
        "RMSE_shots": float(np.sqrt(np.mean(err ** 2))),
        "max_abs_err_shots": abs_err.max(),
        "std_err_shots": err.std(),
        "MAPE_pct": float(np.mean(abs_err / actual) * 100.0),
        "mean_signed_err_shots": err.mean(),
        "R2": r2,
        "pearson_r": pearson,
        "overshoot_pct": 100.0 * over.sum() / n,
        "undershoot_pct": 100.0 * under.sum() / n,
        "exact_pct": 100.0 * exact.sum() / n,
        "mean_overshoot_shots": err[over].mean() if over.any() else 0.0,
        "mean_undershoot_shots": err[under].mean() if under.any() else 0.0,
        "within_1_step_pct": 100.0 * np.mean(abs_err <= 1 * step),
        "within_2_steps_pct": 100.0 * np.mean(abs_err <= 2 * step),
        "within_3_steps_pct": 100.0 * np.mean(abs_err <= 3 * step),
    }
 
 
def analyze_saturation_points(actual_idx, predicted_idx, shot_counts):
    """
    Returns a dict with:
      'metrics'        : metrics over the N averaged circuit-point pairs
      'actual_avg'     : (N,) per-circuit averaged true    shot budget
      'predicted_avg'  : (N,) per-circuit averaged predicted shot budget
    Works for 1-D (N,) or 2-D (N, K) inputs; for (N, K) the K points of each
    circuit are averaged first, so 'metrics' is always over N pairs.
    """
    actual_idx = np.asarray(actual_idx, dtype=int)
    predicted_idx = np.asarray(predicted_idx, dtype=int)
    assert actual_idx.shape == predicted_idx.shape, "arrays must have equal shape"
 
    if actual_idx.ndim == 1:                       # promote (N,) -> (N, 1)
        actual_idx = actual_idx[:, None]
        predicted_idx = predicted_idx[:, None]
    n_circuits, n_points = actual_idx.shape
 
    step = int(shot_counts[1] - shot_counts[0])    # assumes uniform spacing
 
    actual = shot_counts[actual_idx].astype(float)      # (N, K) shot budgets
    predicted = shot_counts[predicted_idx].astype(float)
 
    # average the K near-plateau points of each circuit -> one pair per circuit
    actual_avg = actual.mean(axis=1)                    # (N,)
    predicted_avg = predicted.mean(axis=1)              # (N,)
 
    return {
        "n_circuits": n_circuits,
        "n_points": n_points,
        "step": step,
        "actual_avg": actual_avg,
        "predicted_avg": predicted_avg,
        "metrics": _error_metrics(actual_avg, predicted_avg, step),
    }
 
 
def _print_full_block(m, title):
    line = "-" * 58
    print(f"{title}  (n = {m['n']} pairs)")
    print(line)
    print("Central accuracy")
    print(f"  MAE                      : {m['MAE_shots']:.1f} shots")
    print(f"  Median abs error         : {m['median_abs_err_shots']:.1f} shots")
    print(f"  RMSE                     : {m['RMSE_shots']:.1f} shots")
    print(f"  Max abs error            : {m['max_abs_err_shots']:.1f} shots")
    print(f"  Std of error             : {m['std_err_shots']:.1f} shots")
    print(f"  MAPE                     : {m['MAPE_pct']:.1f} %")
    print("Bias  (+ overshoot / - undershoot)")
    print(f"  Mean signed error        : {m['mean_signed_err_shots']:+.1f} shots")
    print("Agreement")
    print(f"  R^2                      : {m['R2']:.3f}")
    print(f"  Pearson r                : {m['pearson_r']:.3f}")
    print("Over- / under-shoot")
    print(f"  Overshoot                : {m['overshoot_pct']:.1f} %  (mean {m['mean_overshoot_shots']:+.1f})")
    print(f"  Undershoot               : {m['undershoot_pct']:.1f} %  (mean {m['mean_undershoot_shots']:+.1f})")
    print(f"  Exact                    : {m['exact_pct']:.1f} %")
    print("Tolerance accuracy")
    print(f"  Within 1 / 2 / 3 steps   : {m['within_1_step_pct']:.1f} / "
          f"{m['within_2_steps_pct']:.1f} / {m['within_3_steps_pct']:.1f} %")
 
 
def print_report(res, list_circuits=True):
    step = res["step"]
    print("=" * 58)
    print("Saturation-point prediction report")
    print("=" * 58)
    print(f"Test circuits              : {res['n_circuits']}")
    print(f"Saturation points/circuit  : {res['n_points']}  (averaged)")
    print(f"Shot step size             : {step} shots")
    print("=" * 58)
 
    _print_full_block(res["metrics"], "AVERAGED PER CIRCUIT")

hellinger_points = np.array(hellinger_points)
predicted_hellinger_points = np.array(predicted_hellinger_points)

#remove the second dimensions as there is single nosie level for each circuit in the test dataset, so we can flatten the arrays to 1D
hellinger_points = hellinger_points[:, 0, :]
predicted_hellinger_points = predicted_hellinger_points[:, 0, :]

#only take first 5 saturation points for each circuit, as the rest are not relevant for the analysis
hellinger_points = hellinger_points[:, :5]
predicted_hellinger_points = predicted_hellinger_points[:, :5]

print(hellinger_points.shape, predicted_hellinger_points.shape)
metrics = analyze_saturation_points(hellinger_points,
                                        predicted_hellinger_points,
                                        np.arange(50, 1250, 20))
print_report(metrics)

(18, 1) (18, 1)
Saturation-point prediction report
Test circuits              : 18
Saturation points/circuit  : 1  (averaged)
Shot step size             : 20 shots
AVERAGED PER CIRCUIT  (n = 18 pairs)
----------------------------------------------------------
Central accuracy
  MAE                      : 27.8 shots
  Median abs error         : 20.0 shots
  RMSE                     : 35.6 shots
  Max abs error            : 80.0 shots
  Std of error             : 33.4 shots
  MAPE                     : 3.4 %
Bias  (+ overshoot / - undershoot)
  Mean signed error        : -12.2 shots
Agreement
  R^2                      : 0.314
  Pearson r                : 0.664
Over- / under-shoot
  Overshoot                : 22.2 %  (mean +35.0)
  Undershoot               : 55.6 %  (mean -36.0)
  Exact                    : 22.2 %
Tolerance accuracy
  Within 1 / 2 / 3 steps   : 61.1 / 83.3 / 94.4 %


In [8]:
# # Reload train/test datasets
# train_data = np.load('../data/sixth_meeting/ml_training_dataset.npz')
# test_data = np.load('../data/sixth_meeting/ml_testing_dataset.npz')

# X_train_raw, Y_train_raw = train_data['X'], train_data['Y']
# X_test_raw, Y_test_raw = test_data['X'], test_data['Y']

# # Scale targets using train statistics
# scaler_Y_v2 = StandardScaler()
# Y_train_scaled_v2 = scaler_Y_v2.fit_transform(Y_train_raw)
# Y_test_scaled_v2 = scaler_Y_v2.transform(Y_test_raw)

# # Tensors
# X_train_tensor_v2 = torch.tensor(X_train_raw, dtype=torch.float32)
# Y_train_tensor_v2 = torch.tensor(Y_train_scaled_v2, dtype=torch.float32)
# X_test_tensor_v2 = torch.tensor(X_test_raw, dtype=torch.float32)
# Y_test_tensor_v2 = torch.tensor(Y_test_scaled_v2, dtype=torch.float32)

# # DataLoader
# train_dataset_v2 = TensorDataset(X_train_tensor_v2, Y_train_tensor_v2)
# train_loader_v2 = DataLoader(train_dataset_v2, batch_size=4, shuffle=True)

# # New model + same training setup
# model_v2 = HammingDecayPredictor(input_dim=6, output_dim=3)
# criterion_v2 = nn.HuberLoss()
# optimizer_v2 = optim.Adam(model_v2.parameters(), lr=0.001, weight_decay=1e-5)
# epochs_v2 = 1000

# train_loss_curve = []
# test_loss_curve = []

# for epoch_v2 in range(epochs_v2):
#     model_v2.train()
#     running_loss = 0.0

#     for batch_X, batch_Y in train_loader_v2:
#         optimizer_v2.zero_grad()
#         pred = model_v2(batch_X)
#         loss_v2 = criterion_v2(pred, batch_Y)
#         loss_v2.backward()
#         optimizer_v2.step()
#         running_loss += loss_v2.item() * batch_X.size(0)

#     avg_train_loss = running_loss / len(train_loader_v2.dataset)
#     train_loss_curve.append(avg_train_loss)

#     model_v2.eval()
#     with torch.no_grad():
#         test_pred = model_v2(X_test_tensor_v2)
#         avg_test_loss = criterion_v2(test_pred, Y_test_tensor_v2).item()
#     test_loss_curve.append(avg_test_loss)

# # Plot loss curves
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 5))
# plt.plot(train_loss_curve, label='Train Loss')
# plt.plot(test_loss_curve, label='Test Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Huber Loss')
# plt.title('Training vs Test Loss Curve (New HammingDecayPredictor)')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()

In [9]:
# import matplotlib.pyplot as plt

# # Shared shot range for the circuits
# shot_values = shots_dataset[0]

# # Reconstruct Hellinger curves for the training-set targets (A, B, C)
# train_hellinger_curves = np.array([
#     exponential_decay(shot_values, *params)
#     for params in Y_raw
# ])

# # Fitted Hellinger curves for the test-set circuits
# test_hellinger_curves = poly_hellinger[:, 0, :]

# plt.figure(figsize=(12, 6))

# plt.plot([], [], color='tab:blue', label='Training set')
# for curve in train_hellinger_curves:
#     plt.plot(shot_values, curve, color='tab:blue', alpha=0.15, linewidth=1)

# plt.plot([], [], color='tab:orange', label='Test set')
# for curve in test_hellinger_curves:
#     plt.plot(shot_values, curve, color='tab:orange', alpha=0.25, linewidth=1.2)

# plt.xlabel('Shots')
# plt.ylabel('Hellinger value')
# plt.title('Hellinger curves for training and test sets')
# plt.grid(True, alpha=0.3)
# plt.legend()
# plt.show()

In [10]:

circuit_idx = 7

plt.figure(figsize=(10, 5))
plt.plot(shots_dataset[circuit_idx], smoothed_hamming_std[circuit_idx][0], 'o', label='Smoothed Hamming STD')
plt.plot(shots_dataset[circuit_idx], poly_hamming_std[circuit_idx][0], '-', label='Polynomial fit')

plt.title(f"Random circuit: {names_dataset[circuit_idx]}")
plt.xlabel("Shots")
plt.ylabel("Hamming STD")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

NameError: name 'plt' is not defined